## Export CSVs to Database

In [4]:

import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

DB_PATH = Path(os.getenv("DB_PATH"))
CSV_MATCHUPS   = Path("../csvs/matchups/matchups_all_cleaned.csv")
CSV_STANDINGS  = Path("../csvs/season/all_team_standings_cleaned.csv")
CSV_SUMMARIES  = Path("../csvs/season/season_summaries.csv")

engine = create_engine(f"sqlite:///{DB_PATH}")

df_matchups  = pd.read_csv(CSV_MATCHUPS)
df_standings = pd.read_csv(CSV_STANDINGS)
df_summary   = pd.read_csv(CSV_SUMMARIES)

for col in ["Is Home", "Is Bye Week", "IsPlayoff", "IsConsolation"]:
    if col in df_matchups.columns:
        df_matchups[col] = df_matchups[col].astype(int)

df_standings = df_standings.rename(columns={
    "Season": "season",
    "Team ID": "team_id",
    "Team Name": "team_name",
    "Owner(s)_Clean": "owners_clean",
    "Wins": "wins",
    "Losses": "losses",
    "Points For": "points_for",
    "Points Against": "points_against",
    "Regular Season Rank": "regular_season_rank",
    "Final Rank": "final_rank",
})

df_matchups = df_matchups.rename(columns={
    "Season": "season",
    "Week": "week",
    "Team ID": "team_id",
    "Team Name": "team_name",
    "Owner(s)_Clean": "owners_clean",
    "Opponent ID": "opponent_id",
    "Opponent Name": "opponent_name",
    "Score": "score",
    "Opponent Score": "opponent_score",
    "Is Home": "is_home",
    "Is Bye Week": "is_bye_week",
    "IsPlayoff": "is_playoff",
    "IsConsolation": "is_consolation",
})

df_summary = df_summary.rename(columns={
    "Season": "season",
    "Num Teams": "num_teams",
    "Champion Team ID": "champion_team_id",
    "Runner-Up Team ID": "runner_up_team_id",
    "Highest Total Points Team ID": "highest_total_points_team",
    "Average Weekly Score": "avg_weekly_score",
})

with engine.connect() as conn:
    df_standings.to_sql("teams_season", con=conn, if_exists="replace", index=False)
    df_matchups.to_sql("matchups_team", con=conn, if_exists="replace", index=False)
    df_summary.to_sql("season_summaries", con=conn, if_exists="replace", index=False)

engine.dispose()
print("✅ Loaded teams_season, matchups_team, and season_summaries into SQLite")

✅ Loaded teams_season, matchups_team, and season_summaries into SQLite
